In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_Mix
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from network import HierarchicalConvEmbedding, MLP, MixerBlock
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [4]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_Mix()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("reg_dense_0").output,
    outputs=model.output
)

In [8]:
l_label = [1,2,3,4,5,6,7,8,9,12]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_2/Mix-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    #x_attack = np.load(o_path, allow_pickle=True)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_2/Mix-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_2/Mix-8/layer_cache/salt/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_2/Mix-8/layer_cache/base
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_2: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_3: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_4: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_5: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_6: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_7: outputs (20000, 8192), labels (20000,)
[Saved] reg_dense_0: outputs (20000, 128), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data_2/Mix-8/layer_cache/gauss/0
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_2: out

In [11]:
CACHE_DIR = "./Mix-8/w_and_b_cache"

In [12]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_2/Mix-8/layer_cache/base")


==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 17581
xi>=0 count: 15663
xi>=0 count: 13971
xi>=0 count: 13602
xi>=0 count: 14003
xi>=0 count: 12317
xi>=0 count: 11463
xi>=0 count: 13507
xi>=0 count: 16269

==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 18571
xi>=0 count: 17245
xi>=0 count: 15711
xi>=0 count: 14944
xi>=0 count: 15013
xi>=0 count: 14280
xi>=0 count: 13554
xi>=0 count: 14628
xi>=0 count: 16702

==== split 0, thr

In [13]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_2/Mix-8/layer_cache/base")

In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_2/Mix-8/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.84852797 0.69143344 0.58508456 0.58727044 0.60859791 0.48200517
 0.51345093 0.7413094  0.83299851]
Layer 1
accuracy: [0.91884862 0.79338213 0.69601796 0.64254744 0.65930303 0.51711344
 0.48675475 0.62035883 0.7765729 ]
Layer 2
accuracy: [0.92841706 0.81295401 0.71557883 0.65298553 0.66465195 0.53087998
 0.50708325 0.57029072 0.76408915]
Layer 3
accuracy: [0.93360843 0.8207691  0.70341883 0.6624038  0.67344335 0.54244116
 0.49369358 0.62787046 0.77057913]
Layer 4
accuracy: [0.94423325 0.83256477 0.70767206 0.66221078 0.6821739  0.55782563
 0.53316613 0.62766168 0.7679221 ]
Layer 5
accuracy: [0.95281387 0.85155259 0.71271884 0.66679946 0.69141533 0.58908488
 0.54946138 0.6337104  0.77171771]
Layer 6
accuracy: [0.9621824  0.87339776 0.75059878 0.69624016 0.71608195 0.58286417
 0.59575913 0.64623665 0.77617664]
Layer 7
accuracy: [0.96301333 0.90229156 0.77545762 0.74354061 0.73096193 0.70431068
 0.61771856 0.74678252 0.83901209]
Layer 8
accuracy: [0.96300579 0.91987598

In [16]:
compute_stats(base)

(array([[0.70834866, 0.55929117, 0.69591961],
        [0.80274957, 0.6063213 , 0.62789549],
        [0.8189833 , 0.61617249, 0.61382104],
        [0.81926545, 0.6260961 , 0.63071439],
        [0.82815669, 0.6340701 , 0.64291663],
        [0.83902843, 0.64909989, 0.65162983],
        [0.86205964, 0.66506209, 0.67272414],
        [0.88025417, 0.72627107, 0.73450439],
        [0.89887785, 0.71807619, 0.79972886],
        [0.94717996, 0.75715686, 0.79245208],
        [1.        , 0.99961533, 1.        ]]),
 array([[0.10821337, 0.05533871, 0.13434493],
        [0.09121106, 0.0634493 , 0.11843772],
        [0.08699537, 0.06049868, 0.10934403],
        [0.09398053, 0.05932442, 0.11305594],
        [0.09662599, 0.05452551, 0.09644386],
        [0.09841764, 0.04361067, 0.09161622],
        [0.08674992, 0.05868445, 0.07599917],
        [0.07813886, 0.01635543, 0.09075892],
        [0.06271566, 0.04372015, 0.06405218],
        [0.02731074, 0.04801562, 0.07705319],
        [0.        , 0.00022317,

In [17]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/Mix-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.89122908 0.73868215 0.63440307 0.6030073  0.61166149 0.51268066
 0.56401189 0.73243448 0.85470826]
Layer 1
accuracy: [0.93842765 0.82043889 0.71546385 0.64396235 0.64564545 0.57844557
 0.64761792 0.80774956 0.91523193]
Layer 2
accuracy: [0.94128792 0.82703161 0.72766839 0.64804968 0.64451813 0.58939628
 0.6672107  0.79758942 0.91678508]
Layer 3
accuracy: [0.93985115 0.83446192 0.71635566 0.64902351 0.65749039 0.59130274
 0.63264119 0.80334246 0.91305238]
Layer 4
accuracy: [0.94236113 0.84149898 0.71624754 0.65286139 0.66107856 0.59398896
 0.65356201 0.79128747 0.90585646]
Layer 5
accuracy: [0.94505931 0.85077553 0.72011172 0.65991956 0.67161078 0.606269
 0.65125322 0.7781899  0.90102994]
Layer 6
accuracy: [0.94672004 0.85756638 0.74212608 0.67963709 0.69124571 0.60379846
 0.67007432 0.77717124 0.89199843]
Layer 7
accuracy: [0.95064983 0.86950055 0.74962624 0.70196026 0.69181409 0.66754268
 0.6289268  0.78484935 0.88663296]
Layer 8
accuracy: [0.94798145 0.87260328 0

In [18]:
compute_stats(gauss)

(array([[0.75676916, 0.57597447, 0.71719573],
        [0.82389365, 0.62127876, 0.79034125],
        [0.83210998, 0.62608789, 0.79458108],
        [0.82978576, 0.63095879, 0.78342294],
        [0.83229673, 0.63463263, 0.78401849],
        [0.8371767 , 0.64400742, 0.77682135],
        [0.84719736, 0.65444459, 0.7800514 ],
        [0.85578019, 0.68376494, 0.7694577 ],
        [0.86382381, 0.66772017, 0.78752087],
        [0.88944444, 0.66845229, 0.77198513],
        [0.91977438, 0.77384343, 0.87821619]]),
 array([[0.10463715, 0.0475868 , 0.12169489],
        [0.09112581, 0.03206024, 0.10993158],
        [0.08710519, 0.02957224, 0.10151148],
        [0.09145649, 0.02983618, 0.11453614],
        [0.09343451, 0.0317889 , 0.10398122],
        [0.09269712, 0.02879652, 0.10279234],
        [0.08484601, 0.03909652, 0.09017892],
        [0.08172138, 0.01745769, 0.10282108],
        [0.0742672 , 0.04627645, 0.08533278],
        [0.05472863, 0.04103829, 0.08875298],
        [0.04549535, 0.01340247,

In [19]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/Mix-8/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.86790251 0.71038265 0.60064405 0.59270712 0.60885853 0.48477537
 0.52641775 0.74232568 0.84481689]
Layer 1
accuracy: [0.93925803 0.81421913 0.71230648 0.64178885 0.64946286 0.55120131
 0.5705107  0.72124839 0.87873594]
Layer 2
accuracy: [0.94609496 0.8322066  0.72444848 0.65209657 0.65271186 0.55748034
 0.5842055  0.68545068 0.86390605]
Layer 3
accuracy: [0.94487657 0.83732455 0.71327137 0.65384796 0.66401674 0.5664641
 0.54712239 0.71928133 0.85515929]
Layer 4
accuracy: [0.95112814 0.84497241 0.71240988 0.65666344 0.67226648 0.57338301
 0.59096488 0.70793163 0.84364245]
Layer 5
accuracy: [0.95591009 0.8589673  0.71951348 0.66694722 0.67702764 0.59278862
 0.58588056 0.69659611 0.83695594]
Layer 6
accuracy: [0.96003412 0.87278732 0.75148543 0.69099107 0.69160228 0.57754149
 0.62664829 0.7028284  0.82764834]
Layer 7
accuracy: [0.95942524 0.89693611 0.77659835 0.72950415 0.71133714 0.687604
 0.62002794 0.75881218 0.85619471]
Layer 8
accuracy: [0.96191723 0.90666227 0.

In [20]:
compute_stats(salt)

(array([[0.72605099, 0.563757  , 0.70415245],
        [0.82002917, 0.61695928, 0.7235603 ],
        [0.83348322, 0.62376982, 0.71099203],
        [0.83120487, 0.63000819, 0.70824625],
        [0.8350387 , 0.63611064, 0.71214198],
        [0.84385751, 0.64682755, 0.70806471],
        [0.86105999, 0.6576727 , 0.71839559],
        [0.87523136, 0.70780388, 0.74283835],
        [0.88813135, 0.69511429, 0.78850204],
        [0.92137593, 0.70573267, 0.76345391],
        [0.95998235, 0.86713339, 0.93598967]]),
 array([[0.10745671, 0.05109063, 0.13169417],
        [0.09250233, 0.04421137, 0.12519127],
        [0.0891404 , 0.0448658 , 0.11526845],
        [0.09377634, 0.0449635 , 0.12474713],
        [0.09657183, 0.04525672, 0.10523427],
        [0.09561188, 0.03863952, 0.10256416],
        [0.08409884, 0.05368491, 0.08437054],
        [0.07643749, 0.01917915, 0.09893883],
        [0.06647914, 0.05107842, 0.07647654],
        [0.03741789, 0.04824414, 0.09161411],
        [0.02077817, 0.00398522,

In [21]:
SAVE_FILE='e-Mix-8.pkl'

In [22]:
progress = {"base": base, "gauss": gauss,"salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [2]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [3]:
SAVE_FILE='e-Mix-8.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [4]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [5]:
mean_var

{'base': {'mean': array([0.65451981, 0.67898879, 0.68299228, 0.69202531, 0.70171448,
         0.71325272, 0.73328196, 0.78034321, 0.80556097, 0.83226297,
         0.99987178]),
  'std': array([0.1245004 , 0.12855684, 0.13029135, 0.12836032, 0.12333101,
         0.12071171, 0.11782781, 0.09936055, 0.09370704, 0.09902915,
         0.00022245]),
  'min': array([0.48200517, 0.48675475, 0.50708325, 0.49369358, 0.53316613,
         0.54946138, 0.58286417, 0.61771856, 0.65720921, 0.70415263,
         0.99929972]),
  'max': array([0.84852797, 0.91884862, 0.92841706, 0.93360843, 0.94423325,
         0.95281387, 0.9621824 , 0.96301333, 0.96300579, 0.97850711,
         1.        ])},
 'gauss': {'mean': array([0.68331312, 0.74517122, 0.75092632, 0.74805583, 0.75031595,
         0.75266849, 0.76056445, 0.76966761, 0.77302162, 0.77662729,
         0.857278  ]),
  'std': array([0.12394688, 0.12247941, 0.11950951, 0.12112713, 0.11802821,
         0.11477585, 0.10955371, 0.10384286, 0.1072108 , 0.11106